# Entrenar la voz propia de C&M Labs

Este cuaderno afina una voz base en español latino (`es_MX/ald/medium`) con tus grabaciones, y produce el archivo de voz `cm_labs.onnx` que usa el Agente Creador de Video.

**Antes de empezar:**

1. **Activa la tarjeta gráfica:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)** → Guardar.
2. **Sube el conjunto de grabaciones a tu Google Drive:** crea la carpeta **`voz_propia_cm`** en «Mi unidad» y deja dentro el archivo **`dataset_voz_propia.zip`**.
3. Ejecuta las celdas **en orden**, una por una, con el botón ▶ de cada celda (o `Ctrl + Enter`).

**Si Colab te desconecta:** vuelve a ejecutar todas las celdas desde el principio. El avance del entrenamiento se guarda en tu Drive y continúa donde iba.

## 1. Comprobar la tarjeta gráfica

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠ No hay tarjeta gráfica: activa la GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución."

## 2. Conectar tu Google Drive

Te pedirá permiso para acceder a tu Drive: acéptalo con la cuenta donde subiste el archivo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CARPETA = "/content/drive/MyDrive/voz_propia_cm"   # tu carpeta en Drive
ZIP = f"{CARPETA}/dataset_voz_propia.zip"
os.makedirs(f"{CARPETA}/entrenamiento", exist_ok=True)
print("✓ Drive conectado" if os.path.exists(ZIP) else f"⚠ No encuentro {ZIP}. Revisa el nombre de la carpeta y del archivo.")

## 3. Descomprimir las grabaciones

In [ ]:
!rm -rf /content/dataset && mkdir -p /content/dataset && unzip -q "{ZIP}" -d /content/dataset
!echo "Grabaciones: $(ls /content/dataset/wavs | wc -l)" && head -3 /content/dataset/metadata.csv

## 4. Instalar Piper (herramienta de entrenamiento)

Tarda de 5 a 8 minutos y llena la pantalla de mensajes técnicos: es normal. Los avisos en amarillo («performance hint», «FutureWarning») no son errores. Al final debe decir **«✓ Piper listo para entrenar»**.

In [ ]:
%cd /content
!rm -rf piper1-gpl && git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install -q -e ".[train]"
!bash ./build_monotonic_align.sh
# Componente de pronunciación (espeak): se construye con scikit-build, cmake y ninja (3 a 5 minutos).
!python3 -m pip install -q "scikit-build<1" "cmake>=3.18,<4" ninja
!python3 setup.py build_ext --inplace
# Exportar la voz: las versiones recientes de PyTorch piden onnxscript y usan por defecto un método
# de conversión nuevo; las voces de Piper se exportan con el método clásico (dynamo=False).
!python3 -m pip install -q onnxscript
!sed -i 's/model=model_g,/model=model_g, dynamo=False,/' src/piper/train/export_onnx.py
!python3 -c "import piper.espeakbridge, piper.train; print('✓ Piper listo para entrenar')"

## 5. Descargar la voz base (solo la primera vez)

Unos 850 MB. Se guarda en tu Drive para no descargarla de nuevo si Colab te desconecta.

In [ ]:
BASE = f"{CARPETA}/base_es_MX_ald_medium.ckpt"
if not os.path.exists(BASE):
    !wget -q --show-progress -O "{BASE}" "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_MX/ald/medium/epoch%3D9999-step%3D1753600.ckpt"
print("✓ Voz base lista:", round(os.path.getsize(BASE) / 1e6), "MB")

## 6. Entrenar

- **La primera vez** parte de la voz base. **Si ya habías empezado**, continúa desde tu último punto de guardado.
- Entrena en el disco de Colab, que es rápido, y copia a tu Drive **solo el último punto de guardado** (`voz_propia_cm/ultimo.ckpt`, unos 850 MB que se reemplazan cada vez). Así no se llena tu Drive.
- Cada minuto la celda muestra la **época** (vuelta completa a tus grabaciones), la **calidad** medida de 1 a 5 y la hora del último guardado. Con una tarjeta T4, cada época tarda unos 15 segundos y el avance se guarda cada 25 épocas.
- **Mantén la pestaña abierta** y no dejes que el computador se suspenda.


In [ ]:
# 6. Entrenar: trabaja en el disco de Colab (rápido) y copia a tu Drive SOLO el último punto de guardado.
import re, shutil, subprocess, time
import glob, os, sys, zipfile
sys.path.insert(0, "/content/piper1-gpl/src")   # para que esta sesión de Colab encuentre Piper

def puntos_validos():
    """Puntos de guardado completos, del más viejo al más nuevo. Salta los dañados (escritura interrumpida)."""
    candidatos = (glob.glob(f"{CARPETA}/ultimo.ckpt")
                  + glob.glob(f"{CARPETA}/entrenamiento/**/*.ckpt", recursive=True)
                  + glob.glob("/content/entrenamiento/**/*.ckpt", recursive=True))
    return sorted((c for c in candidatos if zipfile.is_zipfile(c)), key=os.path.getmtime)

from IPython.display import clear_output

LOCAL = "/content/entrenamiento"            # disco de Colab: rápido, se borra al desconectar
ULTIMO = f"{CARPETA}/ultimo.ckpt"            # en tu Drive: un solo archivo (~850 MB) que se reemplaza
CONFIG = f"{CARPETA}/cm_labs.onnx.json"
REGISTRO = "/content/entrenamiento.log"
os.makedirs(LOCAL, exist_ok=True)

# ¿Desde dónde se empieza? Desde el punto de guardado completo más reciente; si no hay, desde la voz base.
validos = puntos_validos()
if validos:
    inicio, mensaje = ["--ckpt_path", validos[-1]], f"↺ Continúa desde: {os.path.basename(os.path.dirname(validos[-1]))}/{os.path.basename(validos[-1])}"
else:
    inicio, mensaje = ["--model.warmstart_ckpt", BASE], "▶ Primera vez: se parte de la voz base"

orden = ["python3", "-m", "piper.train", "fit",
         "--data.voice_name", "cm_labs",
         "--data.csv_path", "/content/dataset/metadata.csv",
         "--data.audio_dir", "/content/dataset/wavs",
         "--model.sample_rate", "22050",
         "--data.espeak_voice", "es-419",
         "--data.cache_dir", "/content/cache",
         "--data.config_path", CONFIG,
         "--data.batch_size", "16",
         "--data.num_workers", "2",
         "--trainer.default_root_dir", LOCAL,
         "--trainer.check_val_every_n_epoch", "25",
         "--trainer.max_epochs", "1500", *inicio]

def copiar_a_drive():
    """Copia a Drive el último punto de guardado local, si es más nuevo que el que ya está allá."""
    locales = sorted(glob.glob(f"{LOCAL}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime)
    if locales and zipfile.is_zipfile(locales[-1]) and (not os.path.exists(ULTIMO) or os.path.getmtime(locales[-1]) > os.path.getmtime(ULTIMO)):
        shutil.copy(locales[-1], ULTIMO + ".tmp")
        os.replace(ULTIMO + ".tmp", ULTIMO)
        return True
    return False

with open(REGISTRO, "w") as registro:
    proceso = subprocess.Popen(orden, cwd="/content/piper1-gpl", stdout=registro, stderr=subprocess.STDOUT)

inicio_reloj, ultima_copia = time.time(), "todavía no"
try:
    while proceso.poll() is None:
        time.sleep(60)
        if copiar_a_drive():
            ultima_copia = time.strftime("%H:%M")
        texto = open(REGISTRO, errors="ignore").read()[-20000:]
        epocas = re.findall(r"Epoch (\d+)", texto)
        calidad = re.findall(r"val_mos[:=]\s*([\d.]+)", texto)
        clear_output(wait=True)
        print(mensaje)
        print(f"Época: {epocas[-1] if epocas else '…preparando'} de 1.500"
              f" · minutos entrenando: {(time.time() - inicio_reloj) / 60:.0f}"
              f" · calidad (1 a 5): {calidad[-1] if calidad else 'aún no medida'}"
              f" · último guardado en Drive: {ultima_copia}")
        print("Para escuchar cómo va: pulsa ■ y ejecuta la celda 7. Para seguir después, vuelve a ejecutar esta celda.")
except KeyboardInterrupt:
    print("■ Deteniendo el entrenamiento…")
    proceso.terminate()
    proceso.wait()
finally:
    if copiar_a_drive():
        print("✓ Último punto de guardado copiado a tu Drive (voz_propia_cm/ultimo.ckpt)")
    if proceso.returncode not in (0, None, -15):
        print("⚠ El entrenamiento terminó con un error. Últimas líneas del registro:")
        print(open(REGISTRO, errors="ignore").read()[-3000:])


## 7. Escuchar cómo va

Colab ejecuta **una celda a la vez**: esta celda no corre mientras la 6 está entrenando. Para escuchar:

1. Detén la celda 6 con el botón ■ (mejor después de la época 300).
2. Ejecuta esta celda: convierte el último punto de guardado y lee tres frases que **no** estaban en tus grabaciones.
3. Si todavía suena metálico o raro, vuelve a ejecutar la celda 6: continúa donde iba (se pierden como máximo 25 épocas). Si ya suena como tú, pasa a la celda 8.


In [ ]:
# 7. Escuchar cómo va
import glob, os, sys, zipfile
sys.path.insert(0, "/content/piper1-gpl/src")   # para que esta sesión de Colab encuentre Piper

def puntos_validos():
    """Puntos de guardado completos, del más viejo al más nuevo. Salta los dañados (escritura interrumpida)."""
    candidatos = (glob.glob(f"{CARPETA}/ultimo.ckpt")
                  + glob.glob(f"{CARPETA}/entrenamiento/**/*.ckpt", recursive=True)
                  + glob.glob("/content/entrenamiento/**/*.ckpt", recursive=True))
    return sorted((c for c in candidatos if zipfile.is_zipfile(c)), key=os.path.getmtime)

import wave
from IPython.display import Audio, display

validos = puntos_validos()
if not validos:
    raise SystemExit("⚠ Todavía no hay ningún punto de guardado completo. Deja entrenar la celda 6 al menos 25 épocas.")
ultimo = validos[-1]
print("Punto de guardado usado:", ultimo)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ultimo}" --output-file /content/prueba.onnx
!cp "{CARPETA}/cm_labs.onnx.json" /content/prueba.onnx.json

from piper import PiperVoice
voz = PiperVoice.load("/content/prueba.onnx")
PRUEBAS = [
    "Hola, soy la voz de C y M Labs. Hoy te voy a mostrar cómo analizar las ventas de tu negocio.",
    "Primera alerta: la sede del norte vende un veintiocho por ciento menos desde julio.",
    "¿Quieres que tu equipo trabaje así? Escríbenos y te contamos cómo.",
]
for i, texto in enumerate(PRUEBAS, 1):
    salida = f"/content/prueba_{i}.wav"
    with wave.open(salida, "wb") as w:
        voz.synthesize_wav(texto, w)
    print(texto)
    display(Audio(salida))


## 8. Guardar la voz final en tu Drive

Exporta el último punto de guardado como **`cm_labs.onnx`** (y su configuración **`cm_labs.onnx.json`**) en tu carpeta `voz_propia_cm`. Descárgalos y ponlos juntos en tu computador.

In [ ]:
# 8. Guardar la voz final en tu Drive
import glob, os, sys, zipfile
sys.path.insert(0, "/content/piper1-gpl/src")   # para que esta sesión de Colab encuentre Piper

def puntos_validos():
    """Puntos de guardado completos, del más viejo al más nuevo. Salta los dañados (escritura interrumpida)."""
    candidatos = (glob.glob(f"{CARPETA}/ultimo.ckpt")
                  + glob.glob(f"{CARPETA}/entrenamiento/**/*.ckpt", recursive=True)
                  + glob.glob("/content/entrenamiento/**/*.ckpt", recursive=True))
    return sorted((c for c in candidatos if zipfile.is_zipfile(c)), key=os.path.getmtime)

ultimo = puntos_validos()[-1]
print("Punto de guardado usado:", ultimo)
!cd /content/piper1-gpl && python3 -m piper.train.export_onnx --checkpoint "{ultimo}" --output-file "{CARPETA}/cm_labs.onnx"
print("✓ Voz guardada en tu Drive: voz_propia_cm/cm_labs.onnx y cm_labs.onnx.json")
print('En el agente: config/canal.toml → [voz] modo = "sintetica" y modelo_piper = "ruta/a/cm_labs.onnx"')
